In [ ]:
# STEP 1 — Download Embedding Model
# RUN: First setup only.
# SKIP: If the model has already been downloaded.
#
# Downloads tokenizer.json and model.onnx into:
# models/Xenova/all-MiniLM-L6-v2/

from download import download

download("Xenova/all-MiniLM-L6-v2")

In [ ]:
# STEP 2 — Environment Setup
# RUN: Every time you restart the notebook.
#
# Loads the OpenAI API key from .env and initializes
# the local ONNX embedding model.

from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from sqlitesearch import TextSearchIndex, VectorSearchIndex

from embedder import Embedder

load_dotenv()

openai_client = OpenAI()

embedder = Embedder("models/Xenova/all-MiniLM-L6-v2")

print("Environment initialized successfully.")

In [ ]:
# STEP 3 — Data Preparation & Index Building
# RUN: First setup only, or when rebuilding databases.
# SKIP: If chunks_text.db and chunks_vector.db already exist.
#
# Downloads course lessons, creates overlapping chunks,
# generates embeddings and stores indexes in SQLite.
#
# WARNING: Re-running this step rebuilds the indexes
# and recalculates document embeddings.

from data import load_documents, create_chunks
from search import build_text_index, build_vector_index

documents = load_documents()
chunks = create_chunks(documents)

print("Documents:", len(documents))
print("Chunks:", len(chunks))

build_text_index(chunks)
build_vector_index(chunks, embedder)

print("SQLite indexes built successfully.")

In [ ]:
# STEP 4 — Load Existing SQLite Indexes
# RUN: Every time you restart the notebook.
# SKIP: STEP 3 if the databases already exist.
#
# Loads existing SQLite indexes without recalculating embeddings.

text_index = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="chunks_text.db"
)

vector_index = VectorSearchIndex(
    keyword_fields=["filename"],
    mode="ivf",
    db_path="chunks_vector.db"
)

print("Existing SQLite indexes loaded successfully.")

In [ ]:
# STEP 5 — Hybrid Search Demo
# RUN: Whenever you want to test retrieval.
#
# Combines keyword search and vector search using RRF.
# Returns the five highest-ranked document chunks.
# No OpenAI API call is required.

from search import hybrid_search

query = "How do I give the model access to tools?"

results = hybrid_search(
    text_index,
    vector_index,
    embedder,
    query
)

print("Query:", query)
print("Retrieved chunks:", len(results))

for rank, doc in enumerate(results, start=1):
    print(
        f"{rank}. {doc['filename']} "
        f"(start: {doc['start']})"
    )

In [ ]:
# STEP 6 — Standard RAG Demo
# RUN: Whenever you want to test Standard RAG.
#
# Retrieves relevant chunks and sends them to the LLM
# to generate an answer grounded in course materials.
#
# NOTE: Uses the OpenAI API and consumes tokens.

from rag import rag

question = "How does function calling work?"

rag_result = rag(
    openai_client,
    text_index,
    vector_index,
    embedder,
    question
)

print("Question:", question)
print()
print("Answer:")
print(rag_result["answer"])
print()
print("Input tokens:", rag_result["input_tokens"])

In [ ]:
# STEP 7 — Agentic RAG Demo
# RUN: Whenever you want to test Agentic RAG.
#
# The LLM can call Hybrid Search multiple times
# before generating its final answer.
#
# The agent is limited to five API iterations
# to reduce unnecessary token usage and costs.
#
# NOTE: Uses the OpenAI API and consumes tokens.

from agent import agent_loop

question = (
    "How does the agentic loop work, "
    "and how is it different from plain RAG?"
)

agent_result = agent_loop(
    openai_client,
    text_index,
    vector_index,
    embedder,
    question
)

print("Question:", question)
print()
print("Answer:")
print(agent_result["answer"])
print()
print("Search calls:", agent_result["search_calls"])